In [ ]:
# This example will demonstrate:

    # Defining a Custom Tool: How to create a Python class that encapsulates the logic for interacting with your MCP server.
    # Creating Agents: How to define AI agents with specific roles, goals, and backstories.
    # Defining Tasks: How to set up tasks that your agents will perform, utilizing the custom tools.
    # Orchestrating a Crew: How to bring agents and tasks together into a collaborative crew.

# --- 0. Import CrewAI ---

In [1]:
import os 
# This line imports the os module, which provides a way of using operating system-dependent functionality. 
# One of its common uses is to interact with environment variables.

from crewai import Agent, Task, Crew, LLM
from pydantic import BaseModel, Field
from crewai.tools import BaseTool
from typing import Type
from linkup import LinkupClient

from dotenv import load_dotenv
# Load environment variables from a .env file if it exists.
# This is a good practice for managing API keys and other secrets.
load_dotenv()


True

# --- 0. Import Opik ---

In [2]:
# Import opik and its CrewAI integration
import opik
from opik.integrations.crewai import track_crewai
track_crewai(project_name="arunmanglick-crewai-integration-demo")

# --- 1. Integrate Linkup Search Tool ---

In [3]:
class LinkUpSearchInputDataModel(BaseModel):
    """Input schema for LinkUp Search Tool."""
    query: str = Field(description="The search query to perform")
    depth: str = Field(default="standard",
                       description="Depth of search: 'standard' or 'deep'")
    output_type: str = Field(
        default="searchResults", description="Output type: 'searchResults', 'sourcedAnswer', or 'structured'")
    
# Code Explanation
# The code is a Pydantic data model that defines the input parameters for the LinkUpSearchTool. 
# It acts as a contract, telling the AI agent exactly what information it needs to provide when it decides to use this tool.

# Here's a breakdown of what the code does:

# class LinkUpSearchInput(BaseModel):: This line defines a new class that inherits from BaseModel. 
# Pydantic's BaseModel is used to create data models that automatically validate data types and provide a structured way to handle inputs. 
# In this case, it ensures the LinkUpSearchTool always receives the correct arguments in the correct format.

    # query: str = Field(...): This defines the main search query.
    # query: str specifies that the query field must be a string.
    # Field(description="...") is a Pydantic function that adds metadata to the field. 
    # The description is crucial because it helps the LLM understand what to put in this field. 
    # It's an essential part of how the agent knows what to do.

    # depth: str = Field(...): This defines the depth of the search.
    # depth: str again enforces that the input must be a string.
    # Field(default="standard", ...) sets a default value of "standard". 
    # This means the agent doesn't have to specify this parameter every time; it will automatically use the default unless 
    # it has a reason to change it to "deep".

    # output_type: str = Field(...): This defines the type of output the tool should return.
    # output_type: str ensures the input is a string.
    # Field(default="searchResults", ...) sets a default value, similar to the depth parameter. 
    # The agent will get searchResults by default but has the option to request sourcedAnswer or structured output if its task requires it.

# In short, this Pydantic model provides a clear, documented, and validated schema for the LinkUpSearchTool,
# making it easy for the AI agent to use the tool correctly and for developers to understand its functionality.


In [4]:
# Code Explanation
# This code defines a custom tool class called LinkUpSearchTool, which inherits from a base class BaseTool.
# The purpose of this tool is to perform web searches using the LinkUp API and return comprehensive results.

class LinkUpSearchTool(BaseTool):
    name: str = "LinkUp Search"
    description: str = "Search the web for information using LinkUp and return comprehensive results"
    args_schema: Type[BaseModel] = LinkUpSearchInputDataModel

    def __init__(self):
        super().__init__()

    def _run(self, query: str, depth: str = "standard", output_type: str = "searchResults") -> str:
        """Execute LinkUp search and return results."""
        try:
            # Initialize LinkUp client with API key from environment variables
            linkup_client = LinkupClient(api_key=os.getenv("LINKUP_API_KEY"))

            # Perform search
            search_response = linkup_client.search(
                query=query,
                depth=depth,
                output_type=output_type
            )

            return str(search_response)
        except Exception as e:
            return f"Error occurred while searching: {str(e)}"

# Code Explanation
# This code defines a custom tool class called LinkUpSearchTool, which inherits from a base class BaseTool.
# The purpose of this tool is to perform web searches using the LinkUp API and return comprehensive results.

# The class specifies three attributes:
    # name: A human-readable name for the tool ("LinkUp Search").
    # description: A brief description of what the tool does.
    # args_schema: The expected input data model for the tool, defined elsewhere as LinkUpSearchInputDataModel.

# The constructor (__init__) simply calls the parent class's constructor to ensure proper initialization.
# The core functionality is implemented in the _run method. This method takes a search query, an optional depth parameter (defaulting to "standard"), 
# and an optional output_type (defaulting to "searchResults"). Inside _run, it initializes a LinkupClient using an API key retrieved from the environment variables. 
# It then calls the client's search method with the provided parameters. The results are converted to a string and returned. 
# If any exception occurs during this process, the method catches it and returns an error message.

# A potential "gotcha" here is the reliance on the LINKUP_API_KEY environment variable. If this variable is not set, the search will fail, 
# and the error message will be returned. Also, the code assumes that LinkupClient and LinkUpSearchInputDataModel are properly defined elsewhere in the project.

# --- 1. Setup Local LLM ---

In [5]:
# The LLM class is used to configure a language model.
# We are specifying 'ollama/llama3.2' as the model and pointing to the default local Ollama server address.

local_llm = LLM(
    model="ollama/llama3.2",
    base_url="http://localhost:11434"
)

# What is Ollama
# Ollama is a platform that lets you run LLMs locally on your machine—no cloud dependency required. 
# It’s designed for developers who want fast, private, and customizable access to models like LLaMA, Mistral, Gemma, Phi-4, and more.

# --- 2. Define AI Agent ---

In [ ]:
# An Agent has a specific role, goal, and backstory.
# It also uses the LLM we just configured.
my_research_aiagent = Agent(
     role="Web Search Agent",
    goal="Search the web for information",
    backstory="You are an agent that can search the web for information.",
    verbose=True,
    allow_delegation=False,
    tools=[LinkUpSearchTool()],  # The agent now has the LinkUpSearchTool available
    # llm=local_llm  # Assign the local LLM to this agent
)

# Code Explaination

# role='AI Researcher': This gives the agent its professional identity. 
# The role helps the agent understand its purpose and the type of actions it should take.

# goal='Provide a concise summary of the benefits of using local LLMs':
# This is the agent's main objective. It's the specific task the agent is trying to accomplish within the crew's workflow.

# backstory=(...): 
# This provides context and personality. The backstory helps the agent adopt a specific persona and knowledge base, 
# influencing how it approaches and solves its tasks. In this case, the agent is an "expert AI researcher."

# verbose=True: When set to True, this parameter makes the agent's internal thought process visible. 
# You'll see the agent's reasoning, tool usage, and progress as it works, which is very useful for debugging and 
# understanding its behavior.

# allow_delegation=False: This setting determines whether the agent can pass a task on to another agent in the crew. 
# Here, False means this agent must handle all tasks itself without delegating.


# --- 3. Define the Task ---

In [7]:
# A Task is a specific piece of work for an agent.
# It includes a description, the expected output, and the agent to whom it's assigned.

# research_task = Task(
#     description=(
#         "Research and list at least three key benefits of using a local LLM "
#         "like Ollama over a cloud-based service. "
#         "Focus on aspects like data privacy, cost, and customization. "
#         "The final output should be a clear, concise, and easy-to-read summary."
#     ),
#     expected_output="A summary listing at least three benefits of local LLMs.",
#     agent=my_research_aiagent
# )

research_task = Task(
    description="Research and provide answer to the user's {query} ",
    expected_output="A summary to the user's query",
    agent=my_research_aiagent
)

# --- 4. Create the Crew ---

In [8]:
# A Crew is a collection of agents and their assigned tasks.
# We're using a simple sequential process here, meaning tasks are executed one after the other.

ai_crew = Crew(
    agents=[my_research_aiagent],
    tasks=[research_task],
    verbose=True  # Set verbosity to 2 to see all the agent's thoughts and actions
)

# --- 5. Kick off the crew's work ---

In [9]:
import sys
import os

print("--- Crew is starting its work ---")
result = ai_crew.kickoff({"query": "What is latest news on AI from Sam Altman?"})

print("\n\n--- Crew's final output ---")
print(result)

# Graceful shutdown
sys.stdout.flush()
os._exit(0)

--- Crew is starting its work ---


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 4baef069-dace-47df-b1f3-80844bc3a7ee                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Web Search Agent                                                                                        │
│                                                                                                                 │
│  Task: Research and provide answer to the user's What is latest news on AI from Sam Altman?                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Web Search Agent                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  As of now, Sam Altman, the CEO of OpenAI, has been actively discussing various advancements and developments   │
│  in artificial intelligence. Recently, he participated in discussions about the future of AI regulation,        │
│  emphasizing the importance of balancing innovation with safety and ethical considerations.                     │
│                                                                                                                 │
│  One of the most significant updates from Altman is regarding the ongoing improvements in OpenAI's models,      │
│  particularly with the advancements in their language models and the introduction of new features that enhance  │
│  user interactivity and accessibility. He has also highlighted the collaborative efforts with global leaders    │
│  to establish frameworks for responsible AI deployment, ensuring that the technology benefits society as a      │
│  whole.                                                                                                         │
│                                                                                                                 │
│  Moreover, Altman has been vocal about the need for transparency in AI technologies, advocating for open        │
│  discussions about the implications of AI in various sectors, including healthcare, finance, and education.     │
│  His latest remarks also cover the challenges posed by misinformation and the necessity for robust measures to  │
│  counteract these issues as AI systems become increasingly prevalent in everyday life.                          │
│                                                                                                                 │
│  In summary, Sam Altman continues to be at the forefront of AI conversation, focusing on responsible            │
│  innovation, advancements in technology at OpenAI, and promoting discussions surrounding the ethical impact of  │
│  AI on society.                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 973e0f8b-f7af-496d-ad67-7551f839d567                                                                     │
│  Agent: Web Search Agent                                                                                        │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 4baef069-dace-47df-b1f3-80844bc3a7ee                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: As of now, Sam Altman, the CEO of OpenAI, has been actively discussing various advancements and  │
│  developments in artificial intelligence. Recently, he participated in discussions about the future of AI       │
│  regulation, emphasizing the importance of balancing innovation with safety and ethical considerations.         │
│                                                                                                                 │
│  One of the most significant updates from Altman is regarding the ongoing improvements in OpenAI's models,      │
│  particularly with the advancements in their language models and the introduction of new features that enhance  │
│  user interactivity and accessibility. He has also highlighted the collaborative efforts with global leaders    │
│  to establish frameworks for responsible AI deployment, ensuring that the technology benefits society as a      │
│  whole.                                                                                                         │
│                                                                                                                 │
│  Moreover, Altman has been vocal about the need for transparency in AI technologies, advocating for open        │
│  discussions about the implications of AI in various sectors, including healthcare, finance, and education.     │
│  His latest remarks also cover the challenges posed by misinformation and the necessity for robust measures to  │
│  counteract these issues as AI systems become increasingly prevalent in everyday life.                          │
│                                                                                                                 │
│  In summary, Sam Altman continues to be at the forefront of AI conversation, focusing on responsible            │
│  innovation, advancements in technology at OpenAI, and promoting discussions surrounding the ethical impact of  │
│  AI on society.                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

OPIK: Started logging traces to the "arunmanglick-crewai-integration-demo" project at https://www.comet.com/opik/api/v1/session/redirect/projects/?trace_id=0198e869-3c43-796e-bb7e-8e16fa6318ee&path=aHR0cHM6Ly93d3cuY29tZXQuY29tL29waWsvYXBpLw==.




--- Crew's final output ---
As of now, Sam Altman, the CEO of OpenAI, has been actively discussing various advancements and developments in artificial intelligence. Recently, he participated in discussions about the future of AI regulation, emphasizing the importance of balancing innovation with safety and ethical considerations. 

One of the most significant updates from Altman is regarding the ongoing improvements in OpenAI's models, particularly with the advancements in their language models and the introduction of new features that enhance user interactivity and accessibility. He has also highlighted the collaborative efforts with global leaders to establish frameworks for responsible AI deployment, ensuring that the technology benefits society as a whole.

Moreover, Altman has been vocal about the need for transparency in AI technologies, advocating for open discussions about the implications of AI in various sectors, including healthcare, finance, and education. His latest rema

: 